In [8]:
import os
import glob
import pandas as pd
import numpy as np

DATA_FOLDER = "Signal"
OUTPUT_FOLDER = "Signal_output_hourly"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print("Data folder:", DATA_FOLDER)
print("Output folder:", OUTPUT_FOLDER)


Data folder: Signal
Output folder: Signal_output_hourly


In [9]:
# Load all raw signal tables from the data folder
files = glob.glob(os.path.join(DATA_FOLDER, "*.xlsx"))
print(f"Found {len(files)} files.\n")

Found 276 files.



In [10]:
# Helper functions for column checks
def split_mixed_movements_loose(df):
    """
    TL → 0.5 L + 0.5 T
    TR → 0.5 T + 0.5 R
    If single L/T/R columns do not exist, treat them as 0 and add values.
    """
    directions = [
        "Vehicle_Eastbound",
        "Vehicle_Westbound",
        "Vehicle_Northbound",
        "Vehicle_Southbound",
    ]

    for pref in directions:
        tl = pref + "_TL"
        tr = pref + "_TR"

        # Ensure L/T/R columns exist
        for suffix in ["_L", "_T", "_R"]:
            col = pref + suffix
            if col not in df.columns:
                df[col] = 0

        # TL: allocate 50% to L and 50% to T
        if tl in df.columns:
            df[pref + "_L"] += 0.5 * df[tl]
            df[pref + "_T"] += 0.5 * df[tl]

        # TR: allocate 50% to T and 50% to R
        if tr in df.columns:
            df[pref + "_T"] += 0.5 * df[tr]
            df[pref + "_R"] += 0.5 * df[tr]

    # Remove mixed-lane original columns
    drop_cols = [c for c in df.columns if c.endswith("_TL") or c.endswith("_TR")]
    df = df.drop(columns=drop_cols, errors="ignore")

    return df


def classify_intersection(columns):
    """Classify intersection type based on directional columns present."""
    has_e = any(col.startswith("Vehicle_Eastbound")  for col in columns)
    has_w = any(col.startswith("Vehicle_Westbound")  for col in columns)
    has_n = any(col.startswith("Vehicle_Northbound") for col in columns)
    has_s = any(col.startswith("Vehicle_Southbound") for col in columns)

    count = sum([has_e, has_w, has_n, has_s])

    if count == 4:
        return "4_way"
    elif count == 3:
        return "T_intersection"
    elif count == 2:
        return "2_way"
    else:
        return "other"



In [11]:
directions = {
    "east":  "Vehicle_Eastbound",
    "west":  "Vehicle_Westbound",
    "north": "Vehicle_Northbound",
    "south": "Vehicle_Southbound"
}

all_columns = set()
file_shapes = {}
dfs = []

for f in files:
    try:
        df_raw = pd.read_excel(f, engine="openpyxl")
        cols_raw = df_raw.columns

        # 1. Determine intersection type before splitting TL/TR
        intersection_type = classify_intersection(cols_raw)

        # 2. Now split TL/TR into L/T/R (loose version)
        df = split_mixed_movements_loose(df_raw)

        # 3. Assign intersection type
        df["intersection_type"] = intersection_type

        # 4. Save metadata
        file_shapes[f] = df.shape
        all_columns.update(df.columns)

        dfs.append(df)

    except Exception as e:
        print(f"Error reading {f}: {e}")

In [12]:
# Summary reports
# 1. Column counts for each file
print("Column counts by file:")
for f, shape in file_shapes.items():
    print(f"{os.path.basename(f)}: {shape[1]} columns")

Column counts by file:
tmc_6129022.xlsx: 21 columns
tmc_3255013.xlsx: 21 columns
tmc_8260.xlsx: 20 columns
tmc_5051410.xlsx: 21 columns
tmc_6015002.xlsx: 21 columns
tmc_2051.xlsx: 21 columns
tmc_7013.xlsx: 21 columns
tmc_3512.xlsx: 21 columns
tmc_7678.xlsx: 22 columns
tmc_7547.xlsx: 21 columns
tmc_1240.xlsx: 20 columns
tmc_3553.xlsx: 21 columns
tmc_6301.xlsx: 20 columns
tmc_6028.xlsx: 21 columns
tmc_3089.xlsx: 21 columns
tmc_5018.xlsx: 21 columns
tmc_11150.xlsx: 20 columns
tmc_5262.xlsx: 21 columns
tmc_6115086.xlsx: 21 columns
tmc_1124.xlsx: 21 columns
tmc_6318_partial.xlsx: 21 columns
tmc_6423.xlsx: 21 columns
tmc_7662.xlsx: 21 columns
tmc_7398.xlsx: 21 columns
tmc_5063.xlsx: 21 columns
tmc_11351.xlsx: 20 columns
tmc_7770.xlsx: 20 columns
tmc_1077.xlsx: 21 columns
tmc_5127204.xlsx: 19 columns
tmc_1076.xlsx: 21 columns
tmc_1426.xlsx: 24 columns
tmc_6025.xlsx: 21 columns
tmc_3151024.xlsx: 21 columns
tmc_5127.xlsx: 21 columns
tmc_6313049.xlsx: 20 columns
tmc_2027.xlsx: 21 columns
tmc_773

In [13]:
# 2. Unique columns across all files
print("\nUnique columns across all files:")
for c in sorted(all_columns):
    print(c)

print(f"\nTotal unique columns: {len(all_columns)}")


Unique columns across all files:
-_-_-
Bike_Bike_Total
Bike_Eastbound_T
Bike_Eastbound_Total
Datetime
Exit_Eastbound_L
Exit_Eastbound_R
Exit_Eastbound_T
Exit_Eastbound_Total
Exit_Exit_Total
Exit_Northbound_T
Exit_Northbound_Total
Exit_Southbound_T
Exit_Southbound_Total
Exit_Westbound_L
Exit_Westbound_T
Exit_Westbound_Total
Light_Rail_Transit_Light_Rail_Transit_Total
Light_Rail_Transit_Northbound_L
Light_Rail_Transit_Northbound_Total
SignalID
Vehicle_Eastbound_L
Vehicle_Eastbound_R
Vehicle_Eastbound_T
Vehicle_Eastbound_Total
Vehicle_Northbound_L
Vehicle_Northbound_R
Vehicle_Northbound_T
Vehicle_Northbound_Total
Vehicle_Northeast_Total
Vehicle_Northwest_L
Vehicle_Northwest_Total
Vehicle_Southbound_L
Vehicle_Southbound_R
Vehicle_Southbound_T
Vehicle_Southbound_Total
Vehicle_Vehicle_Total
Vehicle_Westbound_L
Vehicle_Westbound_R
Vehicle_Westbound_T
Vehicle_Westbound_Total
intersection_type

Total unique columns: 42


In [14]:
# Combine all loaded signal tables into one dataframe
all_signals_df = pd.concat(dfs, ignore_index=True)
print(
    all_signals_df[
        [
            "SignalID",
            "intersection_type",
        ]
    ].head(5)
)

   SignalID intersection_type
0   6129022             4_way
1   6129022             4_way
2   6129022             4_way
3   6129022             4_way
4   6129022             4_way


In [15]:
# Build and save metadata table for all signals
meta = all_signals_df[
    [
        "SignalID",
        "intersection_type"
    ]
].drop_duplicates("SignalID")

# Preview metadata
print(meta.head())
print("Total signals:", len(meta))

# Save metadata to file
meta_output_path = os.path.join(OUTPUT_FOLDER, "signal_metadata_split.csv")
meta.to_csv(meta_output_path, index=False)
print("Saved signal metadata to:", meta_output_path)

       SignalID intersection_type
0       6129022             4_way
1008    3255013             4_way
5448       8260    T_intersection
9888    5051410             4_way
14328   6015002             4_way
Total signals: 270
Saved signal metadata to: Signal_output_hourly/signal_metadata_split.csv


In [16]:
# Drop the hour‐label column
if "-_-_-" in all_signals_df.columns:
    all_signals_df = all_signals_df.drop(columns=["-_-_-"])

print("Columns after dropping '-_-_-':")
print(all_signals_df.columns.tolist())

# Convert Datetime to proper datetime type
if "Datetime" not in all_signals_df.columns:
    raise KeyError("Column 'Datetime' not found in data. Please check the file structure.")

all_signals_df["Datetime"] = pd.to_datetime(all_signals_df["Datetime"])

# Create date and day-of-week columns
all_signals_df["hour"] = all_signals_df["Datetime"].dt.hour
all_signals_df["date"] = all_signals_df["Datetime"].dt.date
all_signals_df["dow"] = all_signals_df["Datetime"].dt.weekday    # Monday=0, Sunday=6
all_signals_df["dow_name"] = all_signals_df["Datetime"].dt.day_name()

all_signals_df.head()


Columns after dropping '-_-_-':
['Vehicle_Eastbound_T', 'Vehicle_Eastbound_Total', 'Vehicle_Westbound_T', 'Vehicle_Westbound_Total', 'Vehicle_Northbound_Total', 'Vehicle_Southbound_Total', 'Vehicle_Vehicle_Total', 'SignalID', 'Datetime', 'Vehicle_Eastbound_L', 'Vehicle_Eastbound_R', 'Vehicle_Westbound_L', 'Vehicle_Westbound_R', 'Vehicle_Northbound_L', 'Vehicle_Northbound_T', 'Vehicle_Northbound_R', 'Vehicle_Southbound_L', 'Vehicle_Southbound_T', 'Vehicle_Southbound_R', 'intersection_type', 'Bike_Eastbound_T', 'Bike_Eastbound_Total', 'Bike_Bike_Total', 'Exit_Northbound_T', 'Exit_Northbound_Total', 'Exit_Southbound_T', 'Exit_Southbound_Total', 'Exit_Exit_Total', 'Vehicle_Northeast_Total', 'Light_Rail_Transit_Northbound_L', 'Light_Rail_Transit_Northbound_Total', 'Light_Rail_Transit_Light_Rail_Transit_Total', 'Exit_Eastbound_T', 'Exit_Eastbound_Total', 'Exit_Westbound_T', 'Exit_Westbound_Total', 'Exit_Eastbound_L', 'Vehicle_Northwest_L', 'Vehicle_Northwest_Total', 'Exit_Eastbound_R', 'Exit

,Vehicle_Eastbound_T,Vehicle_Eastbound_Total,Vehicle_Westbound_T,Vehicle_Westbound_Total,Vehicle_Northbound_Total,Vehicle_Southbound_Total,Vehicle_Vehicle_Total,SignalID,Datetime,Vehicle_Eastbound_L,...,Exit_Westbound_Total,Exit_Eastbound_L,Vehicle_Northwest_L,Vehicle_Northwest_Total,Exit_Eastbound_R,Exit_Westbound_L,hour,date,dow,dow_name
0,40.5,55.0,39.5,54.0,0.0,1.0,110.0,6129022,2025-05-01 00:00:00,14.5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2025-05-01,3.0,Thursday
1,28.0,34.0,25.5,29.0,3.0,0.0,66.0,6129022,2025-05-01 01:00:00,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2025-05-01,3.0,Thursday
2,17.0,20.0,21.5,29.0,0.0,1.0,50.0,6129022,2025-05-01 02:00:00,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2025-05-01,3.0,Thursday
3,15.0,22.0,24.0,32.0,1.0,0.0,55.0,6129022,2025-05-01 03:00:00,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2025-05-01,3.0,Thursday
4,63.5,76.0,67.5,78.0,140.0,0.0,294.0,6129022,2025-05-01 04:00:00,12.5,...,NaN,NaN,NaN,NaN,NaN,NaN,4.0,2025-05-01,3.0,Thursday


In [17]:
# Exit_*_Total columns 
exit_total_cols = [c for c in all_signals_df.columns 
                   if c.startswith("Exit_") and c.endswith("Total")]

print("Exit total columns:", exit_total_cols)

# Aggregate exit columns
if exit_total_cols:
    all_signals_df["exit_total"] = all_signals_df[exit_total_cols].sum(axis=1)
else:
    all_signals_df["exit_total"] = 0

all_signals_df[["SignalID", "Datetime", "exit_total"]].head()


Exit total columns: ['Exit_Northbound_Total', 'Exit_Southbound_Total', 'Exit_Exit_Total', 'Exit_Eastbound_Total', 'Exit_Westbound_Total']


,SignalID,Datetime,exit_total
0,6129022,2025-05-01 00:00:00,0.0
1,6129022,2025-05-01 01:00:00,0.0
2,6129022,2025-05-01 02:00:00,0.0
3,6129022,2025-05-01 03:00:00,0.0
4,6129022,2025-05-01 04:00:00,0.0


In [18]:
# Save hourly combined output
hourly_path = os.path.join(OUTPUT_FOLDER, "signal_hourly_combined.csv")
all_signals_df.to_csv(hourly_path, index=False)
print("Saved hourly dataset:", hourly_path)

Saved hourly dataset: Signal_output_hourly/signal_hourly_combined.csv
